# Aula 2 — Setup e Dados

**Intensivão Quant AI — ImpactUFSCar**

---

Hoje você vai ter pela primeira vez os dados que vão alimentar toda a estratégia. Ao final desta aula, você terá:

- Um DataFrame com retornos diários de todas as ações do IBOVESPA
- Entendido a diferença entre retorno simples e log
- Os primeiros plots — curva de preço e curva de retorno

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. O que é uma série temporal de preços

Uma **série temporal** é uma sequência de observações ordenadas no tempo. No nosso caso, cada observação é o preço de fechamento de uma ação em um dia.

```
data         PETR4
2024-01-02   37.50
2024-01-03   38.10
2024-01-04   37.80
...
```

O problema com preços brutos: **eles não são comparáveis entre ativos**. PETR4 a R$ 37 e WEGE3 a R$ 45 não dizem nada sobre qual performou melhor. Para comparar, precisamos de **retornos**.

## 2. Retorno simples vs retorno logarítmico

### Retorno simples

A variação percentual de um dia para o outro:

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

### Retorno logarítmico

O logaritmo natural da razão de preços:

$$r_t^{log} = \ln\left(\frac{P_t}{P_{t-1}}\right) = \ln(P_t) - \ln(P_{t-1})$$

### Por que usamos o log?

Para retornos diários pequenos, os dois são praticamente iguais. Mas o retorno log tem uma propriedade muito útil: **ele é aditivo no tempo**.

Retorno acumulado de dois dias com retorno simples:
$$R_{\text{simples}} = (1 + r_1)(1 + r_2) - 1 \quad \text{(multiplicativo)}$$

Retorno acumulado de dois dias com log:
$$R_{\text{log}} = r_1^{log} + r_2^{log} \quad \text{(aditivo — muito mais simples)}$$

No código abaixo você vai ver que a diferença é pequena no curto prazo, mas cresce com o tempo.

In [ ]:
# Baixamos PETR4 como exemplo para demonstrar os dois tipos de retorno
petr4 = yf.download("PETR4.SA", start="2020-01-01", end="2024-12-31",
                    auto_adjust=True, progress=False)

preco = petr4['Close'].squeeze()

# Calculando os dois retornos
ret_simples = preco.pct_change()
ret_log     = np.log(preco / preco.shift(1))

print("Primeiros 5 dias:")
comparacao = pd.DataFrame({'ret_simples': ret_simples, 'ret_log': ret_log})
print(comparacao.dropna().head())
print(f"\nDiferença média (absoluta): {(ret_simples - ret_log).abs().mean():.6f}")

In [ ]:
# Visualizando: no curto prazo são praticamente iguais
# No longo prazo, o retorno acumulado diverge

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Retornos diários — quase indistinguíveis
ret_simples.dropna().iloc[:60].plot(ax=axes[0], label='simples', alpha=0.8)
ret_log.dropna().iloc[:60].plot(ax=axes[0], label='log', alpha=0.8, linestyle='--')
axes[0].set_title('Retornos diários — 60 primeiros dias')
axes[0].legend()

# Retorno acumulado — divergência aparece
acum_simples = (1 + ret_simples).cumprod() - 1
acum_log     = ret_log.cumsum()

acum_simples.plot(ax=axes[1], label='simples acumulado')
acum_log.plot(ax=axes[1], label='log acumulado', linestyle='--')
axes[1].set_title('Retorno acumulado — 2020 a 2024')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Retorno simples acumulado:", f"{acum_simples.iloc[-1]:.2%}")
print("Retorno log acumulado:    ", f"{np.exp(acum_log.iloc[-1]) - 1:.2%}  (convertido de volta para %)")

## 3. Preços ajustados — dividendos e splits

Olhe o gráfico de preço bruto de uma ação que pagou dividendos. Você vai ver uma queda abrupta na data do pagamento — não porque a empresa foi mal, mas porque o mercado desconta o valor do dividendo do preço.

Se você não ajustar por isso, seu cálculo de retorno vai registrar uma "perda" falsa na data ex-dividendo.

O mesmo acontece com **splits** (quando uma empresa divide cada ação em várias): o preço cai pela metade, mas o acionista não perdeu nada.

**Solução:** usar o **preço ajustado** — o `yfinance` faz esse ajuste automaticamente quando `auto_adjust=True`.

> Usamos sempre `auto_adjust=True` neste intensivão. Nunca calcule retornos sobre preços brutos.

In [ ]:
# Demonstração: preço bruto vs ajustado para VALE3
vale_bruto    = yf.download("VALE3.SA", start="2020-01-01", end="2024-12-31",
                             auto_adjust=False, progress=False)['Close'].squeeze()
vale_ajustado = yf.download("VALE3.SA", start="2020-01-01", end="2024-12-31",
                             auto_adjust=True,  progress=False)['Close'].squeeze()

fig, ax = plt.subplots(figsize=(12, 4))
vale_bruto.plot(ax=ax, label='preço bruto', alpha=0.7)
vale_ajustado.plot(ax=ax, label='preço ajustado', alpha=0.7)
ax.set_title('VALE3 — Bruto vs Ajustado (dividendos incluídos no ajustado)')
ax.legend()
plt.show()

## 4. Baixando todas as ações do IBOVESPA

Agora vamos montar o dataset completo. O IBOVESPA é composto por cerca de 80 ações das empresas mais líquidas da B3. A composição muda a cada quadrimestre.

Abaixo está uma lista dos principais componentes. No intensivão usamos essa lista fixa — para um projeto em produção, você buscaria a composição exata de cada data.

In [ ]:
# Componentes do IBOVESPA (principais ativos, sufixo .SA para B3)
TICKERS_IBOV = [
    'ABEV3.SA', 'ASAI3.SA', 'AZUL4.SA', 'B3SA3.SA', 'BBAS3.SA',
    'BBDC3.SA', 'BBDC4.SA', 'BBSE3.SA', 'BPAC11.SA', 'BRAP4.SA',
    'BRFS3.SA', 'BRKM5.SA', 'CASH3.SA', 'CCRO3.SA', 'CIEL3.SA',
    'CMIG4.SA', 'CMIN3.SA', 'COGN3.SA', 'CPFE3.SA', 'CPLE6.SA',
    'CSAN3.SA', 'CSNA3.SA', 'CYRE3.SA', 'DXCO3.SA', 'EGIE3.SA',
    'ELET3.SA', 'ELET6.SA', 'EMBR3.SA', 'ENEV3.SA', 'ENGI11.SA',
    'EQTL3.SA', 'EZTC3.SA', 'FLRY3.SA', 'GGBR4.SA', 'GOAU4.SA',
    'GOLL4.SA', 'HAPV3.SA', 'HYPE3.SA', 'IGTI11.SA', 'IRBR3.SA',
    'ITSA4.SA', 'ITUB4.SA', 'JBSS3.SA', 'KLBN11.SA', 'LREN3.SA',
    'LWSA3.SA', 'MGLU3.SA', 'MRFG3.SA', 'MRVE3.SA', 'MULT3.SA',
    'NTCO3.SA', 'PCAR3.SA', 'PETR3.SA', 'PETR4.SA', 'PRIO3.SA',
    'QUAL3.SA', 'RADL3.SA', 'RAIL3.SA', 'RDOR3.SA', 'RENT3.SA',
    'RRRP3.SA', 'SANB11.SA', 'SBSP3.SA', 'SLCE3.SA', 'SMTO3.SA',
    'SULA11.SA', 'SUZB3.SA', 'TAEE11.SA', 'TIMS3.SA', 'TOTS3.SA',
    'UGPA3.SA', 'USIM5.SA', 'VALE3.SA', 'VBBR3.SA', 'VIVT3.SA',
    'WEGE3.SA', 'YDUQ3.SA',
]

print(f"Total de tickers: {len(TICKERS_IBOV)}")

In [ ]:
# Download dos preços ajustados — pode levar 1-2 minutos
DATA_INICIO = '2010-01-01'
DATA_FIM    = '2024-12-31'

print("Baixando preços... (aguarde)")
precos_raw = yf.download(
    TICKERS_IBOV,
    start=DATA_INICIO,
    end=DATA_FIM,
    auto_adjust=True,
    progress=True
)['Close']

print(f"\nShape: {precos_raw.shape}  →  {precos_raw.shape[0]} dias × {precos_raw.shape[1]} ações")
print(f"Período: {precos_raw.index[0].date()} a {precos_raw.index[-1].date()}")

## 5. Entendendo o DataFrame

O `precos_raw` é um DataFrame com:
- **Linhas:** dias úteis
- **Colunas:** ações
- **Valores:** preço de fechamento ajustado em reais

Antes de calcular qualquer coisa, precisamos inspecionar os dados.

In [ ]:
# Inspecionando o DataFrame
print("Primeiras linhas:")
print(precos_raw.iloc[:3, :6])  # 3 linhas, 6 primeiras colunas

print("\nÚltimas linhas:")
print(precos_raw.iloc[-3:, :6])

In [ ]:
# Dados ausentes — quantos NaNs por coluna?
nans = precos_raw.isna().sum().sort_values(ascending=False)
print("Ações com mais dados ausentes:")
print(nans[nans > 0].head(10))
print(f"\nTotal de ações sem nenhum NaN: {(nans == 0).sum()}")

**Por que existem NaNs?** Algumas ações do IBOVESPA atual foram listadas depois de 2010 (IPOs). Outras tiveram períodos de suspensão. Na Aula 3 vamos tratar isso com mais cuidado — por hora, seguimos em frente.

## 6. Calculando os retornos

O DataFrame de preços vira um DataFrame de **retornos logarítmicos diários**. Essa será nossa matéria-prima para todo o restante do intensivão.

In [ ]:
# Retornos log diários
# np.log(P_t / P_{t-1}) = np.log(P_t) - np.log(P_{t-1})
retornos = np.log(precos_raw / precos_raw.shift(1))

print(f"Shape retornos: {retornos.shape}")
print(f"NaNs na primeira linha (esperado): {retornos.iloc[0].isna().all()}")

# Removemos a primeira linha (toda NaN — não há retorno no dia 0)
retornos = retornos.iloc[1:]

print(f"\nEstatísticas básicas de PETR4:")
print(retornos['PETR4.SA'].describe())

## 7. Janelas de tempo

No intensivão vamos trabalhar com três frequências:

| Frequência | Uso | Como calcular |
|---|---|---|
| Diária | EDA, visualização | `retornos` (já temos) |
| Mensal | Sinal de momentum, backtest | `retornos.resample('ME').sum()` |
| Anual | Métricas de desempenho | `retornos.resample('YE').sum()` |

Para o **sinal de momentum** (aula 4) vamos usar retornos mensais. Para o **backtest** (aula 5) rebalanceamos mensalmente. Por isso, vale já preparar os dois DataFrames.

In [ ]:
# Retornos mensais — soma dos retornos log diários dentro de cada mês
# (lembre: log é aditivo, então somar é correto)
retornos_mensais = retornos.resample('ME').sum()

print(f"Retornos mensais shape: {retornos_mensais.shape}")
print(f"Período: {retornos_mensais.index[0].date()} a {retornos_mensais.index[-1].date()}")
print()
print("Últimos 3 meses de PETR4 e VALE3:")
print(retornos_mensais[['PETR4.SA', 'VALE3.SA']].tail(3))

## 8. Primeiros plots

Antes de modelar qualquer coisa, olhamos para os dados. Dois plots fundamentais: o preço de uma ação no tempo, e os retornos dela.

In [ ]:
# Plot 1: preço ajustado de algumas ações
acoes_exemplo = ['PETR4.SA', 'VALE3.SA', 'WEGE3.SA', 'ITUB4.SA']

fig, ax = plt.subplots(figsize=(13, 5))

for ticker in acoes_exemplo:
    # Normalizamos para 100 em 2010 para comparar na mesma escala
    serie = precos_raw[ticker].dropna()
    (serie / serie.iloc[0] * 100).plot(ax=ax, label=ticker.replace('.SA', ''))

ax.set_title('Preço ajustado — normalizado em 100 (Jan/2010)')
ax.set_ylabel('Índice (base 100)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: retornos diários de PETR4 — parece ruído, mas tem estrutura
fig, axes = plt.subplots(2, 1, figsize=(13, 7))

retornos['PETR4.SA'].plot(ax=axes[0], alpha=0.7, color='steelblue')
axes[0].set_title('Retornos diários — PETR4 (2010–2024)')
axes[0].set_ylabel('Retorno log')
axes[0].axhline(0, color='black', linewidth=0.8)

retornos_mensais['PETR4.SA'].plot(ax=axes[1], kind='bar',
                                   alpha=0.7, color='steelblue',
                                   width=0.8)
axes[1].set_title('Retornos mensais — PETR4 (2010–2024)')
axes[1].set_ylabel('Retorno log mensal')
axes[1].set_xticklabels(
    [d.strftime('%Y-%m') if i % 12 == 0 else ''
     for i, d in enumerate(retornos_mensais.index)],
    rotation=45
)

plt.tight_layout()
plt.show()

## 9. Salvando os dados

Vamos salvar os DataFrames para não precisar baixar tudo de novo nas próximas aulas.

In [ ]:
precos_raw.to_parquet('precos_ibov.parquet')
retornos.to_parquet('retornos_diarios.parquet')
retornos_mensais.to_parquet('retornos_mensais.parquet')

print("Arquivos salvos:")
print("  precos_ibov.parquet       — preços ajustados diários")
print("  retornos_diarios.parquet  — retornos log diários")
print("  retornos_mensais.parquet  — retornos log mensais")

## Resumo da aula

| O que fizemos | Por que importa |
|---|---|
| Retorno log vs simples | Log é aditivo — facilita cálculos acumulados e é o padrão em finanças quant |
| `auto_adjust=True` | Sem ajuste, dividendos e splits criam retornos falsos |
| DataFrame de retornos | É a matéria-prima de toda a estratégia |
| NaNs identificados | Ações com histórico mais curto — tratar na aula seguinte |
| Retornos mensais | Frequência de rebalanceamento da estratégia de momentum |

---

## Para replicar em casa

1. Rode este notebook do início ao fim
2. Verifique que os três arquivos `.parquet` foram criados
3. Explore: quais ações tiveram o maior e menor retorno acumulado no período?

```python
# Dica: retorno acumulado de cada ação
retorno_acumulado = retornos.sum().sort_values(ascending=False)
print(retorno_acumulado.head(5))   # maiores
print(retorno_acumulado.tail(5))   # menores
```

---

*ImpactUFSCar — Diretoria de Quant*